In [ ]:
!git clone https://github.com/TerrenceReinhardt/thesis_prototype
%cd thesis_prototype

Cloning into 'thesis_prototype'...
remote: Enumerating objects: 108, done.
remote: Counting objects: 100% (108/108), done.
remote: Compressing objects: 100% (79/79), done.
remote: Total 108 (delta 44), reused 88 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (108/108), 578.43 KiB | 6.97 MiB/s, done.
Resolving deltas: 100% (44/44), done.
/content/thesis_prototype


In [ ]:
# ==========================================
# Install Dependencies
# ==========================================

!pip install -q -r requirements.txt
!pip install -q openpyxl
!pip install -q scipy
!pip install -q accelerate
!pip install -q evaluate
!pip install -q transformers datasets scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 123.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 123.6 MB/s eta 0:00:00


In [ ]:
# ==========================================
# Import Libraries
# ==========================================

import os
import json
import random
import shutil
import zipfile
import warnings

import numpy as np
import pandas as pd

import torch

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix
)

from scipy.stats import ttest_rel, wilcoxon

warnings.filterwarnings("ignore")

print("PyTorch :", torch.__version__)
print("CUDA :", torch.cuda.is_available())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device :", device)

PyTorch : 2.11.0+cu128
CUDA : True
Device : cuda


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
# ==========================================
# STEP 6 — Check GPU & Output Directory
# ==========================================

import os
import torch

print("PyTorch version :", torch.__version__)
print("CUDA tersedia   :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU              :", torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        "GPU belum aktif. Buka Runtime > Change runtime type > pilih T4 GPU."
    )

PROJECT_DIR = "/content/thesis_prototype"
OUTPUT_DIR = os.path.join(PROJECT_DIR, "kfold_outputs")

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Output directory :", OUTPUT_DIR)

PyTorch version : 2.11.0+cu128
CUDA tersedia   : True
GPU              : Tesla T4
Project directory: /content/thesis_prototype
Output directory : /content/thesis_prototype/kfold_outputs


In [ ]:
# ==========================================
# STEP 7 — Upload Dataset dari Device
# ==========================================

from google.colab import files
import os

uploaded = files.upload()

if not uploaded:
    raise ValueError("Tidak ada file yang di-upload.")

uploaded_filename = next(iter(uploaded))

DATASET_PATH = os.path.join(
    "/content",
    uploaded_filename
)

print("File berhasil di-upload:")
print(DATASET_PATH)

Saving dataset.xlsx to dataset (3).xlsx
File berhasil di-upload:
/content/dataset (3).xlsx


In [ ]:
import os

print(os.getcwd())

print("\nIsi folder sekarang:")
print(os.listdir("."))

print("\nIsi /content:")
print(os.listdir("/content"))

print("\nIsi thesis_prototype:")
print(os.listdir("/content/thesis_prototype"))


/content/thesis_prototype

Isi folder sekarang:
['app.py', 'requirements.txt', 'src', 'dataset (3).xlsx', '.git', 'dataset.xlsx', 'README.md', '.gitattributes', 'dataset (2).xlsx', 'kfold_outputs', '.gitignore', 'dataset (1).xlsx']

Isi /content:
['.config', 'drive', 'thesis_prototype', 'sample_data']

Isi thesis_prototype:
['app.py', 'requirements.txt', 'src', 'dataset (3).xlsx', '.git', 'dataset.xlsx', 'README.md', '.gitattributes', 'dataset (2).xlsx', 'kfold_outputs', '.gitignore', 'dataset (1).xlsx']


In [ ]:
# ==========================================
# STEP 8 — Load Dataset
# ==========================================

import os
import pandas as pd

DATASET_PATH = "/content/thesis_prototype/dataset (3).xlsx"

if not os.path.exists(DATASET_PATH):
    raise FileNotFoundError(
        f"File tidak ditemukan: {DATASET_PATH}"
    )

file_extension = os.path.splitext(DATASET_PATH)[1].lower()

if file_extension == ".csv":
    try:
        df = pd.read_csv(DATASET_PATH)
    except UnicodeDecodeError:
        df = pd.read_csv(
            DATASET_PATH,
            encoding="latin-1"
        )

elif file_extension in [".xlsx", ".xls"]:
    df = pd.read_excel(DATASET_PATH)

else:
    raise ValueError(
        f"Format file tidak didukung: {file_extension}"
    )

print("Dataset path :", DATASET_PATH)
print("Ukuran data  :", df.shape)
print("Nama kolom   :", df.columns.tolist())

display(df.head(10))

Dataset path : /content/thesis_prototype/dataset (3).xlsx
Ukuran data  : (23644, 4)
Nama kolom   : ['Date', 'User', 'Tweet', 'sentiment']


,Date,User,Tweet,sentiment
0,2022-03-31 21:32:04,pikobar_jabar,Ketahui informasi pembagian #PPKM di wilayah J...,1
1,2022-03-31 16:26:00,inewsdotid,Tempat Ibadah di Wilayah PPKM Level 1 Boleh Be...,1
2,2022-03-31 12:02:34,vdvc_talk,"Juru bicara Satgas Covid-19, Wiku Adisasmito m...",1
3,2022-03-30 21:23:10,pikobar_jabar,Ketahui informasi pembagian #PPKM di wilayah J...,1
4,2022-03-30 18:28:57,tvOneNews,Kementerian Agama menerbitkan Surat Edaran Nom...,1
5,2022-03-30 18:21:18,lampungpost_,"Kapasitas tempat ibadah, termasuk masjid, yang...",1
6,2022-03-30 15:00:19,akusehatklinik,"Halo Sobat Sehat\n\nDengan adanya kondisi ini,...",1
7,2022-03-30 11:35:57,matamilenialID,Mitigasi Penting untuk Cegah Penyebaran Varian...,1
8,2022-03-30 11:21:08,lampungpostid,Sebanyak 5 (lima) kabupaten di Provinsi Lampun...,1
9,2022-03-30 10:16:52,gemaposID,Komentar Satgas Penanganan Covid-19 Tentang Bu...,1


In [ ]:
# ==========================================
# STEP 9 — Inspect Dataset
# ==========================================

print("Ukuran dataset:", df.shape)

print("\nJumlah missing value:")
print(df.isnull().sum())

print("\nDistribusi nilai sentiment:")
print(
    df["sentiment"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nTipe data:")
print(df.dtypes)

Ukuran dataset: (23644, 4)

Jumlah missing value:
Date         0
User         0
Tweet        0
sentiment    0
dtype: int64

Distribusi nilai sentiment:
sentiment
0     1958
1    17706
2     3980
Name: count, dtype: int64

Tipe data:
Date         datetime64[ns]
User                 object
Tweet                object
sentiment             int64
dtype: object


In [ ]:
# ==========================================
# STEP 10 — Prepare Dataset
# ==========================================

import pandas as pd
import numpy as np

TEXT_COLUMN = "Tweet"
LABEL_COLUMN = "sentiment"

working_df = df[
    ["Date", "User", TEXT_COLUMN, LABEL_COLUMN]
].copy()

# Hapus baris yang teks atau labelnya kosong
working_df = working_df.dropna(
    subset=[TEXT_COLUMN, LABEL_COLUMN]
)

# Rapikan teks
working_df[TEXT_COLUMN] = (
    working_df[TEXT_COLUMN]
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Pastikan label berbentuk integer
working_df[LABEL_COLUMN] = pd.to_numeric(
    working_df[LABEL_COLUMN],
    errors="coerce"
)

working_df = working_df.dropna(
    subset=[LABEL_COLUMN]
)

working_df[LABEL_COLUMN] = (
    working_df[LABEL_COLUMN]
    .astype(int)
)

# Hanya gunakan label 0, 1, dan 2
working_df = working_df[
    working_df[LABEL_COLUMN].isin([0, 1, 2])
].reset_index(drop=True)

id2label = {
    0: "negatif",
    1: "positif",
    2: "netral"
}

label2id = {
    "negatif": 0,
    "positif": 1,
    "netral": 2
}

print("Jumlah data setelah cleaning:", len(working_df))

print("\nDistribusi label:")
print(
    working_df[LABEL_COLUMN]
    .map(id2label)
    .value_counts()
)

display(working_df.head())

Jumlah data setelah cleaning: 23644

Distribusi label:
sentiment
positif    17706
netral      3980
negatif     1958
Name: count, dtype: int64


,Date,User,Tweet,sentiment
0,2022-03-31 21:32:04,pikobar_jabar,Ketahui informasi pembagian #PPKM di wilayah J...,1
1,2022-03-31 16:26:00,inewsdotid,Tempat Ibadah di Wilayah PPKM Level 1 Boleh Be...,1
2,2022-03-31 12:02:34,vdvc_talk,"Juru bicara Satgas Covid-19, Wiku Adisasmito m...",1
3,2022-03-30 21:23:10,pikobar_jabar,Ketahui informasi pembagian #PPKM di wilayah J...,1
4,2022-03-30 18:28:57,tvOneNews,Kementerian Agama menerbitkan Surat Edaran Nom...,1


In [ ]:
# ==========================================
# STEP 11 — Formal / Informal Detection
# ==========================================

import re

SLANG_WORDS = {
    "yg", "ga", "gak", "nggak", "enggak", "gk", "gx",
    "gue", "gw", "gua", "lu", "lo", "elo",
    "klo", "kl", "kalo", "kalok",
    "udah", "udh", "dah", "blm", "belom",
    "bgt", "banget", "bener", "beneran",
    "aja", "doang", "kok", "sih", "nih", "tuh",
    "dong", "deh", "lah", "kan",
    "krn", "karna", "karena",
    "dgn", "dg", "sama",
    "utk", "buat",
    "tp", "tapi",
    "jd", "jadi",
    "jgn", "jangan",
    "sm", "sama",
    "dr", "dari",
    "dlm", "dalam",
    "trs", "terus",
    "bs", "bisa",
    "gmn", "gimana",
    "knp", "kenapa",
    "skrg", "sekarang",
    "org", "orang",
    "pake", "pakai",
    "nonton", "liat", "lihat",
    "makasih", "thanks", "thx",
    "wkwk", "wkwkwk", "haha", "hehe",
    "anjir", "anjay", "buset",
    "mantul", "mager", "gabut",
    "otw", "cmiiw", "imo", "lol",
    "ampe", "sampe", "sampek",
    "emang", "emg",
    "kayak", "kaya",
    "biar", "cuman", "cuma",
    "gitu", "gini",
    "tau", "gatau", "gtau",
    "dapet", "dapat",
    "pengen", "pingin",
    "nyari", "nemu",
    "ngapain", "ngeliat",
    "juga", "jg",
    "lebih", "lbh",
    "harus", "hrs",
    "mau", "mo",
    "sudah", "sdh",
    "tidak", "tdk",
    "dulu", "dlu"
}

def tokenize_for_detection(text):
    text = str(text).lower()

    # Ambil kata dan angka saja
    return re.findall(
        r"\b[\w']+\b",
        text
    )

def count_informal_indicators(text):
    tokens = tokenize_for_detection(text)

    slang_count = sum(
        token in SLANG_WORDS
        for token in tokens
    )

    # Contoh karakter berulang:
    # baguuuus, parahhhh, donggg
    repeated_character_count = len(
        re.findall(
            r"(.)\1{2,}",
            str(text).lower()
        )
    )

    return slang_count + repeated_character_count

def classify_language_style(text):
    indicator_count = count_informal_indicators(text)

    if indicator_count >= 2:
        return "informal"

    return "formal"

working_df["informal_indicator_count"] = (
    working_df[TEXT_COLUMN]
    .apply(count_informal_indicators)
)

working_df["language_type"] = (
    working_df[TEXT_COLUMN]
    .apply(classify_language_style)
)

print("Distribusi formal dan informal:")
print(
    working_df["language_type"]
    .value_counts()
)

display(
    working_df[
        [
            TEXT_COLUMN,
            "informal_indicator_count",
            "language_type",
            LABEL_COLUMN
        ]
    ].head(20)
)

Distribusi formal dan informal:
language_type
formal      16876
informal     6768
Name: count, dtype: int64


,Tweet,informal_indicator_count,language_type,sentiment
0,Ketahui informasi pembagian #PPKM di wilayah J...,0,formal,1
1,Tempat Ibadah di Wilayah PPKM Level 1 Boleh Be...,0,formal,1
2,"Juru bicara Satgas Covid-19, Wiku Adisasmito m...",2,informal,1
3,Ketahui informasi pembagian #PPKM di wilayah J...,0,formal,1
4,Kementerian Agama menerbitkan Surat Edaran Nom...,0,formal,1
5,"Kapasitas tempat ibadah, termasuk masjid, yang...",2,informal,1
6,"Halo Sobat Sehat Dengan adanya kondisi ini, Pe...",1,formal,1
7,Mitigasi Penting untuk Cegah Penyebaran Varian...,1,formal,1
8,Sebanyak 5 (lima) kabupaten di Provinsi Lampun...,1,formal,1
9,Komentar Satgas Penanganan Covid-19 Tentang Bu...,0,formal,1


In [ ]:
# ==========================================
# STEP 12 — Split Formal and Informal
# ==========================================

formal_df = (
    working_df[
        working_df["language_type"] == "formal"
    ]
    .copy()
    .reset_index(drop=True)
)

informal_df = (
    working_df[
        working_df["language_type"] == "informal"
    ]
    .copy()
    .reset_index(drop=True)
)

print("Jumlah data formal  :", len(formal_df))
print("Jumlah data informal:", len(informal_df))
print("Total               :", len(formal_df) + len(informal_df))

print("\nDistribusi label formal:")
print(
    formal_df[LABEL_COLUMN]
    .map(id2label)
    .value_counts()
)

print("\nDistribusi label informal:")
print(
    informal_df[LABEL_COLUMN]
    .map(id2label)
    .value_counts()
)

Jumlah data formal  : 16876
Jumlah data informal: 6768
Total               : 23644

Distribusi label formal:
sentiment
positif    14429
netral      1620
negatif      827
Name: count, dtype: int64

Distribusi label informal:
sentiment
positif    3277
netral     2360
negatif    1131
Name: count, dtype: int64


In [ ]:
# ==========================================
# STEP 13 — Save Formal / Informal Dataset
# ==========================================

import os

PROCESSED_DATA_DIR = (
    "/content/thesis_prototype/data/processed"
)

os.makedirs(
    PROCESSED_DATA_DIR,
    exist_ok=True
)

formal_path = os.path.join(
    PROCESSED_DATA_DIR,
    "formal_dataset.csv"
)

informal_path = os.path.join(
    PROCESSED_DATA_DIR,
    "informal_dataset.csv"
)

complete_path = os.path.join(
    PROCESSED_DATA_DIR,
    "dataset_with_language_type.csv"
)

formal_df.to_csv(
    formal_path,
    index=False
)

informal_df.to_csv(
    informal_path,
    index=False
)

working_df.to_csv(
    complete_path,
    index=False
)

print("Dataset formal  :", formal_path)
print("Dataset informal:", informal_path)
print("Dataset lengkap :", complete_path)

Dataset formal  : /content/thesis_prototype/data/processed/formal_dataset.csv
Dataset informal: /content/thesis_prototype/data/processed/informal_dataset.csv
Dataset lengkap : /content/thesis_prototype/data/processed/dataset_with_language_type.csv


In [ ]:
# ==========================================
# Download Processed Dataset
# ==========================================

from google.colab import files

files.download(formal_path)
files.download(informal_path)
files.download(complete_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ==========================================
# STEP 14 — Final Dataset Validation
# ==========================================

import pandas as pd
import numpy as np

TEXT_COLUMN = "Tweet"
LABEL_COLUMN = "sentiment"

for dataset_name, dataset_df in [
    ("formal", formal_df),
    ("informal", informal_df)
]:
    print("=" * 60)
    print(dataset_name.upper())
    print("Jumlah data:", len(dataset_df))

    print("\nDistribusi label:")
    print(
        dataset_df[LABEL_COLUMN]
        .value_counts()
        .sort_index()
    )

    print("\nMissing value:")
    print(
        dataset_df[
            [TEXT_COLUMN, LABEL_COLUMN]
        ].isnull().sum()
    )

    minimum_class_count = (
        dataset_df[LABEL_COLUMN]
        .value_counts()
        .min()
    )

    if minimum_class_count < 5:
        raise ValueError(
            f"Dataset {dataset_name} memiliki kelas dengan "
            f"jumlah kurang dari 5. Tidak dapat menggunakan 5-fold."
        )

FORMAL
Jumlah data: 16876

Distribusi label:
sentiment
0      827
1    14429
2     1620
Name: count, dtype: int64

Missing value:
Tweet        0
sentiment    0
dtype: int64
INFORMAL
Jumlah data: 6768

Distribusi label:
sentiment
0    1131
1    3277
2    2360
Name: count, dtype: int64

Missing value:
Tweet        0
sentiment    0
dtype: int64


In [ ]:
# ==========================================
# STEP 15 — Training Configuration
# ==========================================

import os
import random
import torch
import numpy as np

MODEL_NAME = "indobenchmark/indobert-base-p1"

N_SPLITS = 5
NUM_EPOCHS = 3
MAX_LENGTH = 128

TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32

LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
SEED = 42

KFOLD_OUTPUT_DIR = (
    "/content/thesis_prototype/kfold_training_results"
)

os.makedirs(KFOLD_OUTPUT_DIR, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Model       :", MODEL_NAME)
print("Jumlah fold :", N_SPLITS)
print("Epoch       :", NUM_EPOCHS)
print("GPU aktif   :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU         :", torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        "GPU belum aktif. Pilih Runtime > Change runtime type > T4 GPU."
    )

Model       : indobenchmark/indobert-base-p1
Jumlah fold : 5
Epoch       : 3
GPU aktif   : True
GPU         : Tesla T4


In [ ]:
# ==========================================
# STEP 16 — Load IndoBERT Tokenizer
# ==========================================

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True
)

print("Tokenizer berhasil dimuat.")

config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/229k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Tokenizer berhasil dimuat.


In [ ]:
# ==========================================
# STEP 17 — Metrics and Dataset Functions
# ==========================================

from datasets import Dataset
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

def compute_metrics(eval_prediction):
    predictions = eval_prediction.predictions
    labels = eval_prediction.label_ids

    # Beberapa model dapat mengembalikan tuple
    if isinstance(predictions, tuple):
        predictions = predictions[0]

    predicted_labels = np.argmax(
        predictions,
        axis=-1
    )

    accuracy = accuracy_score(
        labels,
        predicted_labels
    )

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            labels,
            predicted_labels,
            average="macro",
            zero_division=0
        )
    )

    return {
        "accuracy": float(accuracy),
        "macro_precision": float(precision),
        "macro_recall": float(recall),
        "macro_f1": float(f1)
    }


def prepare_huggingface_dataset(dataframe):
    temporary_df = dataframe[
        [TEXT_COLUMN, LABEL_COLUMN]
    ].copy()

    temporary_df = temporary_df.rename(
        columns={
            TEXT_COLUMN: "text",
            LABEL_COLUMN: "labels"
        }
    )

    temporary_df["text"] = (
        temporary_df["text"]
        .astype(str)
        .str.strip()
    )

    temporary_df["labels"] = (
        temporary_df["labels"]
        .astype(int)
    )

    hf_dataset = Dataset.from_pandas(
        temporary_df,
        preserve_index=False
    )

    def tokenize_batch(batch):
        return tokenizer(
            batch["text"],
            truncation=True,
            max_length=MAX_LENGTH
        )

    hf_dataset = hf_dataset.map(
        tokenize_batch,
        batched=True,
        desc="Tokenizing"
    )

    hf_dataset = hf_dataset.remove_columns(
        ["text"]
    )

    return hf_dataset

In [ ]:
# ==========================================
# STEP 18 — Stratified K-Fold Training
# ==========================================

import gc
import json
import shutil
import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    confusion_matrix,
    classification_report
)

from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

id2label = {
    0: "negatif",
    1: "positif",
    2: "netral"
}

label2id = {
    "negatif": 0,
    "positif": 1,
    "netral": 2
}

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)


def train_kfold(dataframe, dataset_name):
    dataset_dir = os.path.join(
        KFOLD_OUTPUT_DIR,
        dataset_name
    )

    if os.path.exists(dataset_dir):
        shutil.rmtree(dataset_dir)

    os.makedirs(dataset_dir, exist_ok=True)

    dataframe = dataframe.copy()

    dataframe = dataframe.dropna(
        subset=[TEXT_COLUMN, LABEL_COLUMN]
    )

    dataframe[TEXT_COLUMN] = (
        dataframe[TEXT_COLUMN]
        .astype(str)
        .str.strip()
    )

    dataframe[LABEL_COLUMN] = (
        dataframe[LABEL_COLUMN]
        .astype(int)
    )

    dataframe = dataframe[
        dataframe[LABEL_COLUMN].isin([0, 1, 2])
    ].reset_index(drop=True)

    splitter = StratifiedKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=SEED
    )

    fold_results = []

    best_fold = None
    best_macro_f1 = -1.0

    texts = dataframe[TEXT_COLUMN].to_numpy()
    labels = dataframe[LABEL_COLUMN].to_numpy()

    for fold, (train_indices, validation_indices) in enumerate(
        splitter.split(texts, labels),
        start=1
    ):
        print("\n" + "=" * 70)
        print(
            f"{dataset_name.upper()} — "
            f"FOLD {fold}/{N_SPLITS}"
        )
        print("=" * 70)

        fold_dir = os.path.join(
            dataset_dir,
            f"fold_{fold}"
        )

        checkpoint_dir = os.path.join(
            fold_dir,
            "checkpoints"
        )

        os.makedirs(fold_dir, exist_ok=True)

        train_df = dataframe.iloc[
            train_indices
        ].reset_index(drop=True)

        validation_df = dataframe.iloc[
            validation_indices
        ].reset_index(drop=True)

        print("Train      :", len(train_df))
        print("Validation :", len(validation_df))

        print("\nDistribusi label validation:")
        print(
            validation_df[LABEL_COLUMN]
            .value_counts()
            .sort_index()
        )

        train_dataset = prepare_huggingface_dataset(
            train_df
        )

        validation_dataset = prepare_huggingface_dataset(
            validation_df
        )

        model = (
            AutoModelForSequenceClassification
            .from_pretrained(
                MODEL_NAME,
                num_labels=3,
                id2label=id2label,
                label2id=label2id
            )
        )

        training_args = TrainingArguments(
            output_dir=checkpoint_dir,

            learning_rate=LEARNING_RATE,
            weight_decay=WEIGHT_DECAY,

            per_device_train_batch_size=(
                TRAIN_BATCH_SIZE
            ),

            per_device_eval_batch_size=(
                EVAL_BATCH_SIZE
            ),

            num_train_epochs=NUM_EPOCHS,

            eval_strategy="epoch",
            save_strategy="epoch",
            logging_strategy="epoch",

            load_best_model_at_end=True,
            metric_for_best_model="macro_f1",
            greater_is_better=True,

            save_total_limit=1,

            fp16=torch.cuda.is_available(),

            report_to="none",

            seed=SEED,
            data_seed=SEED,

            dataloader_num_workers=2
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=validation_dataset,
            processing_class=tokenizer,
            data_collator=data_collator,
            compute_metrics=compute_metrics
        )

        trainer.train()

        evaluation = trainer.evaluate()

        prediction_output = trainer.predict(
            validation_dataset
        )

        logits = prediction_output.predictions

        if isinstance(logits, tuple):
            logits = logits[0]

        predicted_labels = np.argmax(
            logits,
            axis=-1
        )

        actual_labels = prediction_output.label_ids

        fold_metrics = {
            "dataset": dataset_name,
            "fold": fold,
            "train_size": len(train_df),
            "validation_size": len(validation_df),

            "accuracy": float(
                evaluation["eval_accuracy"]
            ),

            "macro_precision": float(
                evaluation["eval_macro_precision"]
            ),

            "macro_recall": float(
                evaluation["eval_macro_recall"]
            ),

            "macro_f1": float(
                evaluation["eval_macro_f1"]
            ),

            "eval_loss": float(
                evaluation["eval_loss"]
            )
        }

        fold_results.append(fold_metrics)

        # Simpan metrik
        with open(
            os.path.join(
                fold_dir,
                "metrics.json"
            ),
            "w",
            encoding="utf-8"
        ) as file:
            json.dump(
                fold_metrics,
                file,
                indent=4
            )

        # Simpan confusion matrix
        cm = confusion_matrix(
            actual_labels,
            predicted_labels,
            labels=[0, 1, 2]
        )

        cm_df = pd.DataFrame(
            cm,
            index=[
                "actual_negatif",
                "actual_positif",
                "actual_netral"
            ],
            columns=[
                "pred_negatif",
                "pred_positif",
                "pred_netral"
            ]
        )

        cm_df.to_csv(
            os.path.join(
                fold_dir,
                "confusion_matrix.csv"
            )
        )

        # Simpan classification report
        report = classification_report(
            actual_labels,
            predicted_labels,
            labels=[0, 1, 2],
            target_names=[
                "negatif",
                "positif",
                "netral"
            ],
            output_dict=True,
            zero_division=0
        )

        with open(
            os.path.join(
                fold_dir,
                "classification_report.json"
            ),
            "w",
            encoding="utf-8"
        ) as file:
            json.dump(
                report,
                file,
                indent=4
            )

        # Simpan prediksi setiap data validation
        prediction_df = validation_df[
            [TEXT_COLUMN, LABEL_COLUMN]
        ].copy()

        prediction_df["actual_label"] = (
            prediction_df[LABEL_COLUMN]
            .map(id2label)
        )

        prediction_df["prediction_id"] = (
            predicted_labels
        )

        prediction_df["predicted_label"] = (
            prediction_df["prediction_id"]
            .map(id2label)
        )

        prediction_df["correct"] = (
            prediction_df[LABEL_COLUMN]
            == prediction_df["prediction_id"]
        )

        prediction_df.to_csv(
            os.path.join(
                fold_dir,
                "predictions.csv"
            ),
            index=False
        )

        current_f1 = fold_metrics["macro_f1"]

        print(
            f"\nAccuracy : "
            f"{fold_metrics['accuracy']:.4f}"
        )

        print(
            f"Macro F1 : "
            f"{fold_metrics['macro_f1']:.4f}"
        )

        # Simpan model dari fold dengan Macro F1 tertinggi
        if current_f1 > best_macro_f1:
            best_macro_f1 = current_f1
            best_fold = fold

            best_model_dir = os.path.join(
                dataset_dir,
                "best_model"
            )

            if os.path.exists(best_model_dir):
                shutil.rmtree(best_model_dir)

            trainer.save_model(best_model_dir)
            tokenizer.save_pretrained(best_model_dir)

        # Hapus checkpoint fold untuk menghemat storage
        if os.path.exists(checkpoint_dir):
            shutil.rmtree(checkpoint_dir)

        del trainer
        del model
        del train_dataset
        del validation_dataset

        gc.collect()
        torch.cuda.empty_cache()

    results_df = pd.DataFrame(fold_results)

    results_df.to_csv(
        os.path.join(
            dataset_dir,
            "all_fold_metrics.csv"
        ),
        index=False
    )

    summary = {
        "dataset": dataset_name,
        "number_of_folds": N_SPLITS,
        "best_fold": int(best_fold),
        "best_macro_f1": float(best_macro_f1)
    }

    metric_columns = [
        "accuracy",
        "macro_precision",
        "macro_recall",
        "macro_f1",
        "eval_loss"
    ]

    for metric in metric_columns:
        summary[f"{metric}_mean"] = float(
            results_df[metric].mean()
        )

        summary[f"{metric}_std"] = float(
            results_df[metric].std(ddof=1)
        )

    with open(
        os.path.join(
            dataset_dir,
            "summary.json"
        ),
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(
            summary,
            file,
            indent=4
        )

    print("\n" + "=" * 70)
    print(f"RINGKASAN {dataset_name.upper()}")
    print("=" * 70)

    for metric in [
        "accuracy",
        "macro_precision",
        "macro_recall",
        "macro_f1"
    ]:
        print(
            f"{metric:16s}: "
            f"{summary[f'{metric}_mean']:.4f} "
            f"± "
            f"{summary[f'{metric}_std']:.4f}"
        )

    print("Best fold       :", best_fold)
    print("Best Macro F1   :", f"{best_macro_f1:.4f}")

    return results_df, summary

In [ ]:
# ==========================================
# STEP 19 — Formal Dataset 5-Fold Training
# ==========================================

formal_results, formal_summary = train_kfold(
    dataframe=formal_df,
    dataset_name="formal"
)

display(formal_results)


FORMAL — FOLD 1/5
Train      : 13500
Validation : 3376

Distribusi label validation:
sentiment
0     166
1    2886
2     324
Name: count, dtype: int64


Tokenizing:   0%|          | 0/13500 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/3376 [00:00<?, ? examples/s]

[transformers] You passed `num_labels=3` which is incompatible to the `id2label` map of length `5`.


pytorch_model.bin:   0%|          | 0.00/498M [00:00<?, ?B/s]

...

Training Loss,Validation Loss,Epoch,Accuracy,Macro Precision,Macro Recall,Macro F1
0.030769,0.223087,3,0.954370,0.874951,0.867435,0.871159



Accuracy : 0.9544
Macro F1 : 0.8712


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


RINGKASAN FORMAL
accuracy        : 0.9509 ± 0.0025
macro_precision : 0.8653 ± 0.0089
macro_recall    : 0.8555 ± 0.0107
macro_f1        : 0.8601 ± 0.0071
Best fold       : 5
Best Macro F1   : 0.8712


,dataset,fold,train_size,validation_size,accuracy,macro_precision,macro_recall,macro_f1,eval_loss
0,formal,1,13500,3376,0.951126,0.862386,0.864406,0.863217,0.247729
1,formal,2,13501,3375,0.950815,0.864743,0.847780,0.856070,0.236319
2,formal,3,13501,3375,0.950815,0.871865,0.842222,0.856394,0.248881
3,formal,4,13501,3375,0.947259,0.852319,0.855683,0.853733,0.242941
4,formal,5,13501,3375,0.954370,0.874951,0.867435,0.871159,0.223087


In [ ]:
# ==========================================
# STEP 20 — Informal Dataset 5-Fold Training
# ==========================================

informal_results, informal_summary = train_kfold(
    dataframe=informal_df,
    dataset_name="informal"
)

display(informal_results)


INFORMAL — FOLD 1/5
Train      : 5414
Validation : 1354

Distribusi label validation:
sentiment
0    226
1    656
2    472
Name: count, dtype: int64


Tokenizing:   0%|          | 0/5414 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/1354 [00:00<?, ? examples/s]

[transformers] You passed `num_labels=3` which is incompatible to the `id2label` map of length `5`.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy,Macro Precision,Macro Recall,Macro F1
0.061025,0.479993,3,0.872136,0.859856,0.848527,0.853852



Accuracy : 0.8721
Macro F1 : 0.8539

RINGKASAN INFORMAL
accuracy        : 0.8784 ± 0.0099
macro_precision : 0.8588 ± 0.0118
macro_recall    : 0.8634 ± 0.0103
macro_f1        : 0.8606 ± 0.0100
Best fold       : 1
Best Macro F1   : 0.8727


,dataset,fold,train_size,validation_size,accuracy,macro_precision,macro_recall,macro_f1,eval_loss
0,informal,1,5414,1354,0.890694,0.871535,0.873891,0.872689,0.438479
1,informal,2,5414,1354,0.884047,0.862339,0.872265,0.866935,0.529436
2,informal,3,5414,1354,0.879616,0.860954,0.862848,0.861888,0.524101
3,informal,4,5415,1353,0.865484,0.839458,0.859403,0.847787,0.518170
4,informal,5,5415,1353,0.872136,0.859856,0.848527,0.853852,0.479993


In [ ]:
# ==========================================
# STEP 21 — Combined Results
# ==========================================

combined_results = pd.concat(
    [
        formal_results,
        informal_results
    ],
    ignore_index=True
)

combined_results.to_csv(
    os.path.join(
        KFOLD_OUTPUT_DIR,
        "combined_fold_results.csv"
    ),
    index=False
)

metrics = [
    "accuracy",
    "macro_precision",
    "macro_recall",
    "macro_f1"
]

summary_rows = []

for dataset_name, result_df in [
    ("formal", formal_results),
    ("informal", informal_results)
]:
    row = {
        "dataset": dataset_name
    }

    for metric in metrics:
        mean_value = result_df[metric].mean()
        std_value = result_df[metric].std(ddof=1)

        row[f"{metric}_mean"] = mean_value
        row[f"{metric}_std"] = std_value

        row[f"{metric}_mean_std"] = (
            f"{mean_value:.4f} ± {std_value:.4f}"
        )

    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)

summary_df.to_csv(
    os.path.join(
        KFOLD_OUTPUT_DIR,
        "mean_std_summary.csv"
    ),
    index=False
)

display(combined_results)
display(summary_df)

,dataset,fold,train_size,validation_size,accuracy,macro_precision,macro_recall,macro_f1,eval_loss
0,formal,1,13500,3376,0.951126,0.862386,0.864406,0.863217,0.247729
1,formal,2,13501,3375,0.950815,0.864743,0.847780,0.856070,0.236319
2,formal,3,13501,3375,0.950815,0.871865,0.842222,0.856394,0.248881
3,formal,4,13501,3375,0.947259,0.852319,0.855683,0.853733,0.242941
4,formal,5,13501,3375,0.954370,0.874951,0.867435,0.871159,0.223087
5,informal,1,5414,1354,0.890694,0.871535,0.873891,0.872689,0.438479
6,informal,2,5414,1354,0.884047,0.862339,0.872265,0.866935,0.529436
7,informal,3,5414,1354,0.879616,0.860954,0.862848,0.861888,0.524101
8,informal,4,5415,1353,0.865484,0.839458,0.859403,0.847787,0.518170
9,informal,5,5415,1353,0.872136,0.859856,0.848527,0.853852,0.479993


,dataset,accuracy_mean,accuracy_std,accuracy_mean_std,macro_precision_mean,macro_precision_std,macro_precision_mean_std,macro_recall_mean,macro_recall_std,macro_recall_mean_std,macro_f1_mean,macro_f1_std,macro_f1_mean_std
0,formal,0.950877,0.002518,0.9509 ± 0.0025,0.865253,0.008853,0.8653 ± 0.0089,0.855505,0.010697,0.8555 ± 0.0107,0.860115,0.007116,0.8601 ± 0.0071
1,informal,0.878396,0.009882,0.8784 ± 0.0099,0.858829,0.011775,0.8588 ± 0.0118,0.863387,0.010322,0.8634 ± 0.0103,0.860630,0.009971,0.8606 ± 0.0100


In [ ]:
# ==========================================
# STEP 22 — Significance Testing
# ==========================================

from scipy.stats import (
    ttest_ind,
    mannwhitneyu
)

significance_rows = []

for metric in metrics:
    formal_scores = (
        formal_results[metric]
        .to_numpy()
    )

    informal_scores = (
        informal_results[metric]
        .to_numpy()
    )

    # Welch independent t-test
    t_statistic, t_p_value = ttest_ind(
        formal_scores,
        informal_scores,
        equal_var=False
    )

    # Mann–Whitney U
    u_statistic, u_p_value = mannwhitneyu(
        formal_scores,
        informal_scores,
        alternative="two-sided"
    )

    significance_rows.append({
        "metric": metric,

        "formal_mean": formal_scores.mean(),
        "informal_mean": informal_scores.mean(),

        "mean_difference": (
            formal_scores.mean()
            - informal_scores.mean()
        ),

        "welch_t_statistic": t_statistic,
        "welch_t_p_value": t_p_value,

        "mann_whitney_u": u_statistic,
        "mann_whitney_p_value": u_p_value,

        "significant_welch_0.05": (
            t_p_value < 0.05
        ),

        "significant_mann_whitney_0.05": (
            u_p_value < 0.05
        )
    })

significance_df = pd.DataFrame(
    significance_rows
)

significance_df.to_csv(
    os.path.join(
        KFOLD_OUTPUT_DIR,
        "significance_test_results.csv"
    ),
    index=False
)

display(significance_df)

,metric,formal_mean,informal_mean,mean_difference,welch_t_statistic,welch_t_p_value,mann_whitney_u,mann_whitney_p_value,significant_welch_0.05,significant_mann_whitney_0.05
0,accuracy,0.950877,0.878396,0.072481,15.893803,0.000039,25.0,0.011925,True,True
1,macro_precision,0.865253,0.858829,0.006424,0.975140,0.360174,19.0,0.222222,False,False
2,macro_recall,0.855505,0.863387,-0.007882,-1.185650,0.269822,7.0,0.309524,False,False
3,macro_f1,0.860115,0.860630,-0.000516,-0.094118,0.927569,12.0,1.000000,False,False


In [ ]:
# ==========================================
# STEP 23 — Automatic Interpretation
# ==========================================

interpretation = []

for _, row in significance_df.iterrows():
    metric_name = row["metric"]

    interpretation.append(
        f"METRIK: {metric_name.upper()}"
    )

    interpretation.append(
        f"Rata-rata formal: "
        f"{row['formal_mean']:.4f}"
    )

    interpretation.append(
        f"Rata-rata informal: "
        f"{row['informal_mean']:.4f}"
    )

    interpretation.append(
        f"Welch t-test p-value: "
        f"{row['welch_t_p_value']:.6f}"
    )

    if row["welch_t_p_value"] < 0.05:
        interpretation.append(
            "Kesimpulan: terdapat perbedaan "
            "yang signifikan antara data formal "
            "dan informal."
        )
    else:
        interpretation.append(
            "Kesimpulan: tidak terdapat perbedaan "
            "yang signifikan antara data formal "
            "dan informal."
        )

    interpretation.append(
        f"Mann-Whitney p-value: "
        f"{row['mann_whitney_p_value']:.6f}"
    )

    interpretation.append("-" * 60)

interpretation_text = "\n".join(
    interpretation
)

print(interpretation_text)

with open(
    os.path.join(
        KFOLD_OUTPUT_DIR,
        "significance_interpretation.txt"
    ),
    "w",
    encoding="utf-8"
) as file:
    file.write(interpretation_text)


METRIK: ACCURACY
Rata-rata formal: 0.9509
Rata-rata informal: 0.8784
Welch t-test p-value: 0.000039
Kesimpulan: terdapat perbedaan yang signifikan antara data formal dan informal.
Mann-Whitney p-value: 0.011925
------------------------------------------------------------
METRIK: MACRO_PRECISION
Rata-rata formal: 0.8653
Rata-rata informal: 0.8588
Welch t-test p-value: 0.360174
Kesimpulan: tidak terdapat perbedaan yang signifikan antara data formal dan informal.
Mann-Whitney p-value: 0.222222
------------------------------------------------------------
METRIK: MACRO_RECALL
Rata-rata formal: 0.8555
Rata-rata informal: 0.8634
Welch t-test p-value: 0.269822
Kesimpulan: tidak terdapat perbedaan yang signifikan antara data formal dan informal.
Mann-Whitney p-value: 0.309524
------------------------------------------------------------
METRIK: MACRO_F1
Rata-rata formal: 0.8601
Rata-rata informal: 0.8606
Welch t-test p-value: 0.927569
Kesimpulan: tidak terdapat perbedaan yang signifikan antara d

In [ ]:
# ==========================================
# STEP 24 — Export Complete Results to Excel
# ==========================================

excel_path = os.path.join(
    KFOLD_OUTPUT_DIR,
    "kfold_complete_results.xlsx"
)

with pd.ExcelWriter(
    excel_path,
    engine="openpyxl"
) as writer:
    formal_results.to_excel(
        writer,
        sheet_name="Formal Folds",
        index=False
    )

    informal_results.to_excel(
        writer,
        sheet_name="Informal Folds",
        index=False
    )

    summary_df.to_excel(
        writer,
        sheet_name="Mean Std Summary",
        index=False
    )

    significance_df.to_excel(
        writer,
        sheet_name="Significance Test",
        index=False
    )

print("File Excel:")
print(excel_path)

File Excel:
/content/thesis_prototype/kfold_training_results/kfold_complete_results.xlsx


In [ ]:
# ==========================================
# STEP 25 — ZIP and Download Results
# ==========================================

import shutil
import os
from google.colab import files

ZIP_BASE_PATH = "/content/Thesis_KFold_Results"

zip_path = shutil.make_archive(
    ZIP_BASE_PATH,
    "zip",
    KFOLD_OUTPUT_DIR
)

zip_size_mb = (
    os.path.getsize(zip_path)
    / (1024 ** 2)
)

print("ZIP berhasil dibuat:")
print(zip_path)
print(f"Ukuran ZIP: {zip_size_mb:.2f} MB")

files.download(zip_path)

ZIP berhasil dibuat:
/content/Thesis_KFold_Results.zip
Ukuran ZIP: 883.36 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>